# DC 親子関係 最適化モデル

仕入れ先への発注最小ロット $Q$ が大きい場合、全 DC が直接仕入れると平均在庫が $Q/2$ になり在庫コストが膨らむ。
DC 間で **親子関係** を設定すると、子 DC は週 1 回・需要量分だけ補充でき平均在庫が $d/2$ まで減る。

このモデルは **在庫削減メリット** と **DC 間輸送コスト増加** を天秤にかけ、週次コスト合計が最小になる親子関係を求める。

```
仕入れ先 ─[タリフA]─> 親DC ─[タリフB]─> 子DC
仕入れ先 ─[タリフA]─> 孤立DC
```

## 1. DC の役割

| 役割 | 仕入れ元 | ロット | 子を持つか |
|------|---------|--------|------------|
| 孤立 DC | 仕入れ先（直接） | Q（大） | なし |
| 親 DC   | 仕入れ先（直接） | Q（大） | あり |
| 子 DC   | 親 DC（経由）   | 週間需要量（小） | なし |

各 DC の役割は最適化によって決定される（事前に固定しない）。

## 2. 数理モデル

### パラメータ

| 記号 | 意味 | 単位 |
|------|------|------|
| $d_j$ | DC $j$ の週間需要量 | ケース/週 |
| $h_j$ | DC $j$ の在庫保管単価 | 円/ケース/日 |
| $Q_j$ | DC $j$ のロットサイズ | ケース |
| $a_j$ | 仕入れ先 → DC $j$ のタリフ A | 円/ケース |
| $b_{jk}$ | DC $j$ → DC $k$ のタリフ B（非対称） | 円/ケース |

### 決定変数

$$x_{jk} \in \{0, 1\} \quad (j \neq k)$$

$x_{jk} = 1$ のとき、DC $k$ を DC $j$ の子として割り当てる。

### 目的関数（週次コスト最小化）

**孤立 DC / 親 DC**（仕入れ先から直接補充）:

$$C_j^{\text{direct}} = 7 h_j \frac{Q_j}{2} + a_j \left(d_j + \sum_k x_{jk} d_k\right)$$

**子 DC**（親 DC から補充）:

$$C_j^{\text{child}} = 7 h_j \frac{d_j}{2} + \sum_l x_{lj} b_{lj} d_j$$

> DC 間輸送コストは受け取り側（子 DC）に帰属する。

**全体の目的関数**:

$$\min \sum_j \left[ \left(1 - \sum_l x_{lj}\right) C_j^{\text{direct}} + \left(\sum_l x_{lj}\right) C_j^{\text{child}} \right]$$

### 制約条件

| # | 内容 | 式 |
|---|------|-----|
| C1 | 各 DC の親は 1 つまで | $\sum_j x_{jk} \leq 1 \quad \forall k$ |
| C2 | 2 層固定（親 DC は子 DC になれない） | $x_{lj} + x_{jk} \leq 1 \quad \forall l, j, k$ |
| C3 | 自己割り当て禁止 | $x_{jj} = 0 \quad \forall j$ |

## 3. 入出力仕様

### 入力 Excel ファイル構成

**シート「DCマスタ」**

| 列 | 型 | 説明 |
|----|----|------|
| `dc_id` | str | DC 識別子 |
| `demand` | float | 週間需要量（ケース/週） |
| `holding_cost` | float | 在庫保管単価（円/ケース/日） |
| `lot_size` | float | ロットサイズ（ケース） |
| `tariff_from_supplier` | float | タリフ A（円/ケース） |

**シート「DC間タリフ」**

| 列 | 型 | 説明 |
|----|----|------|
| `from_dc_id` | str | 送り元 DC |
| `to_dc_id` | str | 送り先 DC |
| `tariff` | float | タリフ B（円/ケース） |

### 出力

**① トップレベル**

| フィールド | 型 | 説明 |
|-----------|---|------|
| `status` | str | `"Optimal"` / `"Infeasible"` / `"Unbounded"` |
| `total_weekly_cost` | float | 最適化後の週次コスト合計（円/週） |
| `parent_of` | dict | `{dc_id: parent_dc_id \| None}` |
| `cost_breakdown` | list | コスト内訳（②参照） |
| `summary` | dict | DC 別サマリー DataFrame（⑤参照） |

**② `cost_breakdown`**（行のリスト — 縦: 倉庫、横: コスト区分）

| フィールド | 型 | 説明 |
|-----------|---|------|
| `scenario` | str | `"baseline"` = 全倉庫孤立 / `"optimized"` = 最適化後 |
| `dc_id` | str \| None | 倉庫 ID（合計行は `None`） |
| `holding_cost` | float | 在庫コスト（円/週） |
| `supplier_transport_cost` | float | 仕入れ先→DC 輸送コスト（円/週） |
| `dc_transport_cost` | float | DC 間輸送コスト（円/週） |
| `total` | float | 上記合計（円/週） |

**③ コスト帰属ルール**

| 役割 | 在庫コスト | 仕入れ先→DC 輸送 | DC 間輸送 |
|------|-----------|----------------|----------|
| 孤立 DC | $7 h_j Q_j / 2$ | $a_j d_j$ | 0 |
| 親 DC | $7 h_j Q_j / 2$ | $a_j (d_j + \sum_k x_{jk} d_k)$ | 0 |
| 子 DC | $7 h_j d_j / 2$ | 0 | $b_{\text{parent},j} \cdot d_j$ |

**⑤ `summary`（DC 別サマリー DataFrame）**

`summary["baseline"]`（親子設定なし）と `summary["optimized"]`（最適化後）の 2 つの DataFrame。
各 DC を 1 行とし、マスタ情報・役割・週次コストを横に並べる。

| 列名 | 型 | 説明 | 単位 |
|------|----|------|------|
| `dc_id` | str | DC 識別子 | — |
| `demand` | float | 週間需要量 $d_j$ | ケース/週 |
| `holding_cost_unit` | float | 在庫保管単価 $h_j$ | 円/ケース/日 |
| `lot_size` | float | ロットサイズ $Q_j$ | ケース |
| `tariff_supplier` | float | タリフ A $a_j$ | 円/ケース |
| `role` | str | `"孤立"` / `"親"` / `"子"` | — |
| `parent_dc` | str \| None | 親 DC の ID。孤立・親 DC は `None` | — |
| `holding_cost` | float | 在庫コスト（週次） | 円/週 |
| `supplier_transport_cost` | float | 仕入れ先→DC 輸送コスト（週次） | 円/週 |
| `dc_transport_cost` | float | DC 間輸送コスト（週次） | 円/週 |
| `total_cost` | float | 週次コスト合計 | 円/週 |

---
## 4. セットアップ

In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from IPython.display import display
from src.excel_loader import load_excel
from src.optimizer import solve

%matplotlib inline
plt.rcParams['font.family'] = ['Yu Gothic', 'MS Gothic', 'Meiryo', 'DejaVu Sans']

---
## 5. Excel からデータを読み込む

`data/sample.xlsx` にはシナリオ 1 のサンプルデータが入っています。

In [12]:
dcs, dc_tariffs = load_excel("data/sample2.xlsx")

print("=== DCマスタ ===")
display(pd.DataFrame(dcs))

=== DCマスタ ===


,dc_id,demand,holding_cost,lot_size,tariff_from_supplier
0,北海道,50,10,500,300
1,東北,50,10,500,200
2,北関東,150,10,500,150
3,関東,120,10,500,150
4,西関東,100,10,500,150


In [13]:
print("=== DC間タリフ ===")
display(pd.DataFrame(dc_tariffs))

=== DC間タリフ ===


,from_dc_id,to_dc_id,tariff
0,北海道,東北,300
1,北海道,北関東,400
2,北海道,関東,400
3,北海道,西関東,400
4,東北,北海道,300
5,東北,北関東,300
6,東北,関東,300
7,東北,西関東,300
8,北関東,北海道,400
9,北関東,東北,300


---
## 6. 最適化を実行する

In [14]:
result = solve(dcs, dc_tariffs)
result

{'status': 'Optimal',
 'total_weekly_cost': 161500.0,
 'parent_of': {'北海道': '関東', '東北': '関東', '北関東': None, '関東': None, '西関東': None}}

---
## 7. 結果の確認

In [ ]:
_SCENARIO_LABELS = {
    "baseline":  "ベースライン（全孤立）",
    "optimized": "最適化後",
}


def plot_cost_comparison(cost_breakdown, title="倉庫別コスト比較"):
    """cost_breakdown リストから積み上げ棒グラフを描画する。"""
    bl_rows   = [r for r in cost_breakdown if r["scenario"] == "baseline"  and r["dc_id"] is not None]
    opt_rows  = [r for r in cost_breakdown if r["scenario"] == "optimized" and r["dc_id"] is not None]
    bl_total  = next(r for r in cost_breakdown if r["scenario"] == "baseline"  and r["dc_id"] is None)
    opt_total = next(r for r in cost_breakdown if r["scenario"] == "optimized" and r["dc_id"] is None)

    dc_ids = [r["dc_id"] for r in bl_rows]
    groups = dc_ids + ["【合計】"]
    n = len(groups)

    components = [
        ("holding_cost",            "在庫コスト",       "#4C72B0"),
        ("supplier_transport_cost", "仕入れ先→DC 輸送", "#DD8452"),
        ("dc_transport_cost",       "DC 間輸送",        "#55A868"),
    ]

    fig, ax = plt.subplots(figsize=(max(8, n * 2.4), 5))
    x = np.arange(n)
    w = 0.35

    for side, (rows, total_row, alpha, hatch) in enumerate([
        (bl_rows,  bl_total,  1.0,  ""),
        (opt_rows, opt_total, 0.65, "///"),
    ]):
        bottoms = np.zeros(n)
        for key, comp_label, color in components:
            vals = np.array([r[key] for r in rows] + [total_row[key]])
            ax.bar(
                x + (side - 0.5) * w, vals, w,
                bottom=bottoms, color=color, alpha=alpha, hatch=hatch,
                label=comp_label if side == 0 else "_nolegend_",
            )
            bottoms += vals

        totals = np.array([r["total"] for r in rows] + [total_row["total"]])
        for xi, tot in zip(x + (side - 0.5) * w, totals):
            ax.text(xi, tot * 1.01, f"{tot:,.0f}", ha="center", va="bottom", fontsize=8)

    ax.set_xticks(x)
    ax.set_xticklabels(groups)
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))
    ax.set_ylabel("週次コスト（円/週）")
    ax.set_title(f"{title}\n左: ベースライン（全孤立）  /  右: 最適化後")
    ax.legend(title="コスト内訳", fontsize=9)
    plt.tight_layout()
    display(fig)
    plt.close(fig)


def _display_cost_table(cost_breakdown):
    rows = [
        {
            "シナリオ": _SCENARIO_LABELS[r["scenario"]],
            "倉庫": r["dc_id"] if r["dc_id"] is not None else "【合計】",
            "在庫コスト": r["holding_cost"],
            "仕入れ先→DC 輸送": r["supplier_transport_cost"],
            "DC 間輸送": r["dc_transport_cost"],
            "合計": r["total"],
        }
        for r in cost_breakdown
    ]
    df = pd.DataFrame(rows).set_index(["シナリオ", "倉庫"])
    display(df.style.format("{:,.0f}"))


def _display_summary(summary):
    """SPEC 4.2⑤ DC 別サマリー DataFrame を baseline / optimized に分けて表示する。"""
    cost_cols = ["holding_cost", "supplier_transport_cost", "dc_transport_cost", "total_cost"]
    fmt = {c: "{:,.0f}" for c in cost_cols}

    for label, df in [("baseline（親子設定なし）", summary["baseline"]),
                      ("optimized（最適化後）",    summary["optimized"])]:
        print(f"\n--- {label} ---")
        disp = df.copy()
        disp["parent_dc"] = disp["parent_dc"].fillna("—")
        display(disp.style.format(fmt).set_properties(subset=cost_cols, **{"text-align": "right"}))
        total = df[cost_cols].sum()
        print(
            f"  合計  "
            f"在庫={total['holding_cost']:>10,.0f}  "
            f"仕入れ先輸送={total['supplier_transport_cost']:>10,.0f}  "
            f"DC間輸送={total['dc_transport_cost']:>10,.0f}  "
            f"合計={total['total_cost']:>10,.0f}"
        )


def display_result(result):
    print(f"ステータス  : {result['status']}")
    print(f"週次コスト  : {result['total_weekly_cost']:,.0f} 円/週")
    print()
    print("親子関係:")
    for dc_id, parent in result["parent_of"].items():
        role = "孤立 DC / 親 DC" if parent is None else f"子 DC（親: {parent}）"
        print(f"  {dc_id}: {role}")

    if result.get("cost_breakdown"):
        print()
        _display_cost_table(result["cost_breakdown"])
        plot_cost_comparison(result["cost_breakdown"])

    if result.get("summary"):
        _display_summary(result["summary"])

display_result(result)

---
## 8. シナリオ検証（SPEC.md 準拠）

SPEC.md に記載の 3 シナリオで期待値と一致するかを確認する。

In [ ]:
def run_scenario(title, dcs_data, tariffs_data, expected_cost, expected_parent_of):
    print(f"{'=' * 55}")
    print(f"  {title}")
    print(f"{'=' * 55}")
    res = solve(dcs_data, tariffs_data)
    display_result(res)

    cost_ok   = abs(res["total_weekly_cost"] - expected_cost) < 0.01
    parent_ok = res["parent_of"] == expected_parent_of
    verdict   = "PASS" if cost_ok and parent_ok else "FAIL"
    print(f"\n期待コスト: {expected_cost:,.0f} 円/週  [{verdict}]\n")

### シナリオ 1: 親子関係を作る方が最適（2 DC）

DC-B のタリフ A が高いため、DC-A を親にして DC-B を子にする構成が最適になる。
在庫削減効果（31,500 → 3,500 円/週）が輸送コスト増加（+400 円/週）を大きく上回る。

| パターン | DC-A | DC-B | 合計 |
|---------|------|------|------|
| 両方孤立 | 17,750 | 18,250 | **36,000** |
| DC-B が DC-A の子 | 18,000 | 1,900 | **19,900** ← 最適 |
| DC-A が DC-B の子 | 1,900 | 19,000 | **20,900** |

In [ ]:
run_scenario(
    "シナリオ 1: 親子関係を作る方が最適（2 DC）",
    dcs_data=[
        {"dc_id": "DC-A", "demand": 50, "holding_cost": 10, "lot_size": 500, "tariff_from_supplier": 5},
        {"dc_id": "DC-B", "demand": 50, "holding_cost": 10, "lot_size": 500, "tariff_from_supplier": 15},
    ],
    tariffs_data=[
        {"from_dc_id": "DC-A", "to_dc_id": "DC-B", "tariff": 3},
        {"from_dc_id": "DC-B", "to_dc_id": "DC-A", "tariff": 3},
    ],
    expected_cost=19900.0,
    expected_parent_of={"DC-A": None, "DC-B": "DC-A"},
)

### シナリオ 2: 全 DC 孤立が最適（2 DC）

DC 間タリフ B が非常に高い（70 円/ケース）ため、
在庫削減効果（3,150 円/週）を輸送コスト増加（3,500 円/週）が上回り、全孤立が最適になる。

| パターン | DC-A | DC-B | 合計 |
|---------|------|------|------|
| 両方孤立 | 3,750 | 3,750 | **7,500** ← 最適 |
| DC-B が DC-A の子 | 4,000 | 3,850 | **7,850** |
| DC-A が DC-B の子 | 3,850 | 4,000 | **7,850** |

In [ ]:
run_scenario(
    "シナリオ 2: 全 DC 孤立が最適（2 DC）",
    dcs_data=[
        {"dc_id": "DC-A", "demand": 50, "holding_cost": 2, "lot_size": 500, "tariff_from_supplier": 5},
        {"dc_id": "DC-B", "demand": 50, "holding_cost": 2, "lot_size": 500, "tariff_from_supplier": 5},
    ],
    tariffs_data=[
        {"from_dc_id": "DC-A", "to_dc_id": "DC-B", "tariff": 70},
        {"from_dc_id": "DC-B", "to_dc_id": "DC-A", "tariff": 70},
    ],
    expected_cost=7500.0,
    expected_parent_of={"DC-A": None, "DC-B": None},
)

### シナリオ 3: DC-C を親にした集約が最適（3 DC）

DC-C は保管単価 h=2 が低く、大ロット在庫コストが小さい。
DC-C ↔ 他の輸送タリフは高い（70 円/ケース）が、DC-A・DC-B の在庫削減効果（各 14,000 円/週）がそれを上回る。

| パターン | 合計 |
|---------|------|
| 全孤立 | **41,000** |
| DC-B が DC-A の子・DC-C 孤立 | **26,300** |
| DC-A・DC-B が DC-C の子 | **26,000** ← 最適 |

In [ ]:
run_scenario(
    "シナリオ 3: DC-C を親にした集約が最適（3 DC）",
    dcs_data=[
        {"dc_id": "DC-A", "demand": 100, "holding_cost": 10, "lot_size": 500, "tariff_from_supplier": 5},
        {"dc_id": "DC-B", "demand": 100, "holding_cost": 10, "lot_size": 500, "tariff_from_supplier": 15},
        {"dc_id": "DC-C", "demand": 100, "holding_cost": 2,  "lot_size": 500, "tariff_from_supplier": 5},
    ],
    tariffs_data=[
        {"from_dc_id": "DC-A", "to_dc_id": "DC-B", "tariff": 3},
        {"from_dc_id": "DC-B", "to_dc_id": "DC-A", "tariff": 3},
        {"from_dc_id": "DC-A", "to_dc_id": "DC-C", "tariff": 70},
        {"from_dc_id": "DC-C", "to_dc_id": "DC-A", "tariff": 70},
        {"from_dc_id": "DC-B", "to_dc_id": "DC-C", "tariff": 70},
        {"from_dc_id": "DC-C", "to_dc_id": "DC-B", "tariff": 70},
    ],
    expected_cost=26000.0,
    expected_parent_of={"DC-A": "DC-C", "DC-B": "DC-C", "DC-C": None},
)